In [1]:
import subprocess
from moviepy import VideoFileClip

def robust_load_video(path, step=0.5, max_try=10.0):
    """
    Intenta cargar un video. Si falla, intenta saltar 'step' segundos 
    hasta un máximo de 'max_try'.
    """
    t_start = 0.0
    while t_start <= max_try:
        try:
            # Intentamos abrir el video. 
            # Si t_start > 0, usamos subclip para saltar el posible error inicial.
            clip = VideoFileClip(path)
            if t_start > 0:
                clip = clip.subclip(t_start)
            
            # Forzamos una lectura rápida de un frame para verificar que funciona
            clip.get_frame(0) 
            return clip, t_start
        except Exception:
            t_start += step
            print(f"  --> Fallo en t={t_start-step}s, reintentando en t={t_start}s...")
    
    return None, None

In [2]:
import os
import cv2
from moviepy import VideoFileClip
import librosa
import numpy as np
from skimage.metrics import structural_similarity as ssim

# -------------------------
# Parámetros
# -------------------------
DATA_PATH = "../data/EXIST 2026 Videos Dataset/training/videos/"
OUTPUT_PATH = "../data/important_images/"
MIN_DISTANCE_SEC = 2
NUM_PICOS = 3
SSIM_THRESHOLD = 0.3
DURATION = 2
DARK_THRESHOLD = 30

# -------------------------
# Función tiempo bonito
# -------------------------
def format_time(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"

# -------------------------
# Procesar videos
# -------------------------
for path in os.listdir(DATA_PATH):
    video_path = os.path.join(DATA_PATH, path)
    video_name = os.path.splitext(path)[0]  # nombre del video sin extensión
    print(f"\nProcesando video: {video_path}")

    # Crear carpeta para guardar imágenes
    save_dir = os.path.join(OUTPUT_PATH, video_name)
    save_dir = os.path.join(OUTPUT_PATH, video_name)
    if os.path.exists(save_dir):
        if len(os.listdir(save_dir)) > 0:
            print(f"Saltando video {video_name}, ya tiene imágenes.")
            continue  # pasa al siguiente video
        else:
            print(f"Reprocesando video {video_name}, carpeta vacía.")
    else:
        os.makedirs(save_dir, exist_ok=True)

    video, start_used = robust_load_video(video_path)
    
    if video is None:
        print(f"❌ Video {path} definitivamente ilegible tras varios intentos.")
        continue
    
    if start_used > 0:
        print(f"⚠️ Video cargado saltando los primeros {start_used} segundos.")

    # Audio
    audio = video.audio.to_soundarray()
    sr = video.audio.fps
    y = audio.mean(axis=1)  # convertir a mono

    # RMS
    frame_length = 2048
    hop_length = 512
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]

    # Detectar picos de audio
    indices = np.argsort(rms)[::-1]
    selected = []
    min_distance = int(MIN_DISTANCE_SEC * sr / hop_length)

    for idx in indices:
        if all(abs(idx - s) > min_distance for s in selected):
            selected.append(idx)
        if len(selected) >= 10:
            break

    # Convertir a tiempos
    selected_times = [(idx * hop_length / sr, rms[idx]) for idx in selected]

    # Filtrar frames similares y descartar escenas oscuras
    final_frames = []

    for time_sec, rms_val in selected_times:
        frame = video.get_frame(time_sec)
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

        # Descartar escenas oscuras, pero permitir al menos uno
        if gray.mean() < DARK_THRESHOLD and len(final_frames) > 0:
            continue

        added = True
        for i, (existing_t, existing_frame, existing_rms) in enumerate(final_frames):
            existing_gray = cv2.cvtColor(existing_frame, cv2.COLOR_RGB2GRAY)
            score = ssim(gray, existing_gray)
            if score > SSIM_THRESHOLD:
                if rms_val > existing_rms:
                    final_frames[i] = (time_sec, frame, rms_val)
                added = False
                break

        if added:
            final_frames.append((time_sec, frame, rms_val))

    # Si final_frames quedó vacío (todos muy oscuros), tomar el frame con mayor RMS
    if len(final_frames) == 0 and len(selected_times) > 0:
        time_sec, rms_val = max(selected_times, key=lambda x: x[1])
        frame = video.get_frame(time_sec)
        final_frames.append((time_sec, frame, rms_val))

    # Tomar solo los primeros NUM_PICOS frames finales
    final_frames = final_frames[:NUM_PICOS]

    # Guardar frames como imágenes
    for i, (time_sec, frame, _) in enumerate(final_frames):
        save_path = os.path.join(save_dir, f"pico{i+1}.jpg")
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)  # cv2 usa BGR
        cv2.imwrite(save_path, frame_bgr)
        print(f"Guardado: {save_path}")


Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6959705969173499142.mp4
Saltando video 6959705969173499142, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7058005271284845871.mp4
Saltando video 7058005271284845871, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7134731486275652870.mp4
Saltando video 7134731486275652870, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7099478288724135173.mp4
Saltando video 7099478288724135173, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7307715406486162721.mp4
Saltando video 7307715406486162721, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7005267923912772869.mp4
Saltando video 7005267923912772869, ya tiene imágenes.

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7208329845049281797.mp4
Saltando video 7208

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/7298026909340814625.mp4, 1769472 bytes wanted but 0 bytes read at frame index 0 (out of a total 280 frames), at time 0.00/11.20 sec. Using the last valid frame instead.
  warnings.warn(


  --> Fallo en t=0.0s, reintentando en t=0.5s...
  --> Fallo en t=0.5s, reintentando en t=1.0s...
  --> Fallo en t=1.0s, reintentando en t=1.5s...
  --> Fallo en t=1.5s, reintentando en t=2.0s...
  --> Fallo en t=2.0s, reintentando en t=2.5s...
  --> Fallo en t=2.5s, reintentando en t=3.0s...
  --> Fallo en t=3.0s, reintentando en t=3.5s...
  --> Fallo en t=3.5s, reintentando en t=4.0s...
  --> Fallo en t=4.0s, reintentando en t=4.5s...
  --> Fallo en t=4.5s, reintentando en t=5.0s...
  --> Fallo en t=5.0s, reintentando en t=5.5s...
  --> Fallo en t=5.5s, reintentando en t=6.0s...
  --> Fallo en t=6.0s, reintentando en t=6.5s...
  --> Fallo en t=6.5s, reintentando en t=7.0s...
  --> Fallo en t=7.0s, reintentando en t=7.5s...
  --> Fallo en t=7.5s, reintentando en t=8.0s...
  --> Fallo en t=8.0s, reintentando en t=8.5s...
  --> Fallo en t=8.5s, reintentando en t=9.0s...
  --> Fallo en t=9.0s, reintentando en t=9.5s...
  --> Fallo en t=9.5s, reintentando en t=10.0s...
  --> Fallo en t=10

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Guardado: ../data/important_images/7087331129929256197/pico1.jpg
Guardado: ../data/important_images/7087331129929256197/pico2.jpg
Guardado: ../data/important_images/7087331129929256197/pico3.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7039086371855797510.mp4
Guardado: ../data/important_images/7039086371855797510/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6938863068973665542.mp4
Guardado: ../data/important_images/6938863068973665542/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6904949752836738309.mp4
Guardado: ../data/important_images/6904949752836738309/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6963323211740155141.mp4
Guardado: ../data/important_images/6963323211740155141/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6800566129891740934.mp4
Guardado: ../data/important_images/6800566129891740934/pico1.jpg

Procesando vid

...

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6957860609790676230.mp4, 1869696 bytes wanted but 0 bytes read at frame index 297 (out of a total 879 frames), at time 9.90/29.30 sec. Using the last valid frame instead.
  warnings.warn(
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6957860609790676230.mp4, 1869696 bytes wanted but 0 bytes read at frame index 237 (out of a total 879 frames), at time 7.90/29.30 sec. Using the last valid frame instead.
  warnings.warn(
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6957860609790676230.mp4, 18

Guardado: ../data/important_images/6957860609790676230/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7120430768089861381.mp4
Guardado: ../data/important_images/7120430768089861381/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6936649725361851654.mp4
Guardado: ../data/important_images/6936649725361851654/pico1.jpg
Guardado: ../data/important_images/6936649725361851654/pico2.jpg
Guardado: ../data/important_images/6936649725361851654/pico3.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6882930146802781441.mp4
Guardado: ../data/important_images/6882930146802781441/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6903674386214112513.mp4
Guardado: ../data/important_images/6903674386214112513/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7076627834990611754.mp4
Guardado: ../data/important_images/7076627834990611754/pico1.jpg

Procesando vid

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6843540294646893829.mp4, 1769472 bytes wanted but 0 bytes read at frame index 341 (out of a total 1034 frames), at time 11.37/34.47 sec. Using the last valid frame instead.
  warnings.warn(
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6843540294646893829.mp4, 1769472 bytes wanted but 0 bytes read at frame index 281 (out of a total 1034 frames), at time 9.37/34.47 sec. Using the last valid frame instead.
  warnings.warn(
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/6843540294646893829.mp4,

Guardado: ../data/important_images/6843540294646893829/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6989299791121403141.mp4
Guardado: ../data/important_images/6989299791121403141/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6934322858030075141.mp4
Guardado: ../data/important_images/6934322858030075141/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6954397180337655045.mp4
Guardado: ../data/important_images/6954397180337655045/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/6875395840479972609.mp4
Guardado: ../data/important_images/6875395840479972609/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7194102098739399941.mp4
Guardado: ../data/important_images/7194102098739399941/pico1.jpg

Procesando video: ../data/EXIST 2026 Videos Dataset/training/videos/7072726688118967557.mp4
Guardado: ../data/important_images/70727266881189675

KeyboardInterrupt: 

In [2]:
import os
import cv2
import librosa
import numpy as np
from moviepy import VideoFileClip
from skimage.metrics import structural_similarity as ssim
from concurrent.futures import ProcessPoolExecutor
import functools

# -------------------------
# Parámetros Globales
# -------------------------
DATA_PATH = "../data/EXIST 2026 Videos Dataset/training/videos/"
OUTPUT_PATH = "../data/important_images/"
MIN_DISTANCE_SEC = 2
NUM_PICOS = 3
SSIM_THRESHOLD = 0.3
DARK_THRESHOLD = 30
MAX_WORKERS = 12  # Ajusta esto según tu CPU (ej. la mitad de tus núcleos)

def robust_load_video(path, step=0.5, max_try=2.0):
    """ Intenta cargar el video saltando el inicio si está corrupto. """
    t_start = 0.0
    while t_start <= max_try:
        try:
            clip = VideoFileClip(path)
            if t_start > 0:
                clip = clip.subclip(t_start)
            clip.get_frame(0) 
            return clip, t_start
        except Exception:
            t_start += step
    return None, None

def process_single_video(filename):
    """ Función que procesa un solo video. """
    video_path = os.path.join(DATA_PATH, filename)
    video_name = os.path.splitext(filename)[0]
    save_dir = os.path.join(OUTPUT_PATH, video_name)

    # 1. Verificación de existencia
    if os.path.exists(save_dir) and len(os.listdir(save_dir)) > 0:
        return f"Saltando {video_name}: ya procesado."

    os.makedirs(save_dir, exist_ok=True)

    # 2. Carga Robusta
    video, _ = robust_load_video(video_path)
    if video is None:
        return f"Error: {video_name} es ilegible."

    try:
        # 3. Audio y RMS
        audio_clip = video.audio
        if audio_clip is None:
            return f"Error: {video_name} no tiene audio."
        
        audio_array = audio_clip.to_soundarray()
        sr = audio_clip.fps
        y = audio_array.mean(axis=1)
        
        hop_length = 512
        rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]

        # 4. Selección de picos
        indices = np.argsort(rms)[::-1]
        selected = []
        min_distance = int(MIN_DISTANCE_SEC * sr / hop_length)

        for idx in indices:
            if all(abs(idx - s) > min_distance for s in selected):
                selected.append(idx)
            if len(selected) >= 10: break

        selected_times = [(idx * hop_length / sr, rms[idx]) for idx in selected]

        # 5. Filtrado Visual (SSIM)
        final_frames = []
        for time_sec, rms_val in selected_times:
            # Asegurarse de no pedir un tiempo mayor a la duración
            if time_sec >= video.duration: continue
            
            frame = video.get_frame(time_sec)
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

            if gray.mean() < DARK_THRESHOLD and len(final_frames) > 0:
                continue

            added = True
            for i, (existing_t, existing_frame, existing_rms) in enumerate(final_frames):
                existing_gray = cv2.cvtColor(existing_frame, cv2.COLOR_RGB2GRAY)
                # Redimensionar para SSIM si los frames no coinciden por algún error de lectura
                if gray.shape != existing_gray.shape:
                    existing_gray = cv2.resize(existing_gray, (gray.shape[1], gray.shape[0]))
                
                score = ssim(gray, existing_gray)
                if score > SSIM_THRESHOLD:
                    if rms_val > existing_rms:
                        final_frames[i] = (time_sec, frame, rms_val)
                    added = False
                    break
            if added:
                final_frames.append((time_sec, frame, rms_val))

        # 6. Guardado
        final_frames = final_frames[:NUM_PICOS]
        for i, (time_sec, frame, _) in enumerate(final_frames):
            save_path = os.path.join(save_dir, f"pico{i+1}.jpg")
            cv2.imwrite(save_path, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        # Limpieza CRÍTICA para evitar fugas de memoria y procesos zombie
        video.close()
        if video.audio: video.audio.close()
        
        return f"Éxito: {video_name} procesado."

    except Exception as e:
        return f"Error procesando {video_name}: {str(e)}"

# -------------------------
# Ejecución Principal
# -------------------------
if __name__ == "__main__":
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    all_videos = [f for f in os.listdir(DATA_PATH) if f.endswith(('.mp4', '.avi', '.mov'))]
    
    print(f"Iniciando procesamiento en paralelo para {len(all_videos)} videos...")
    print(f"Usando {MAX_WORKERS} procesos simultáneos.")

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(process_single_video, all_videos))

    for res in results:
        print(res)

Iniciando procesamiento en paralelo para 2524 videos...
Usando 12 procesos simultáneos.


/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/7298026909340814625.mp4, 1769472 bytes wanted but 0 bytes read at frame index 0 (out of a total 280 frames), at time 0.00/11.20 sec. Using the last valid frame instead.
  warnings.warn(
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file ../data/EXIST 2026 Videos Dataset/training/videos/7172562634015886597.mp4, 1769472 bytes wanted but 0 bytes read at frame index 0 (out of a total 809 frames), at time 0.00/28.30 sec. Using the last valid frame instead.
  warnings.warn(


Saltando 6959705969173499142: ya procesado.
Saltando 7058005271284845871: ya procesado.
Saltando 7134731486275652870: ya procesado.
Saltando 7099478288724135173: ya procesado.
Saltando 7307715406486162721: ya procesado.
Saltando 7005267923912772869: ya procesado.
Saltando 7208329845049281797: ya procesado.
Saltando 7177128274924014891: ya procesado.
Saltando 7025996781834030342: ya procesado.
Saltando 7188786155016441130: ya procesado.
Saltando 6940414220282465541: ya procesado.
Saltando 7156028727980199174: ya procesado.
Saltando 6996470929987620101: ya procesado.
Saltando 6857235524772531462: ya procesado.
Saltando 7003732227335015686: ya procesado.
Saltando 6971946065121627397: ya procesado.
Saltando 6947429811841256710: ya procesado.
Saltando 6978836546979958021: ya procesado.
Saltando 6939179911286410502: ya procesado.
Saltando 6983408743987875078: ya procesado.
Saltando 7118380022066334978: ya procesado.
Saltando 6915604005347511557: ya procesado.
Saltando 7016082700259921157: ya